In [ ]:
import pickle
import os
import pandas as pd
import asyncio
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm
from kaggle_secrets import UserSecretsClient

async def translate_text(text, client, semaphore, max_retries=6):
    async with semaphore:
        for attempt in range(max_retries):
            try:
                response = await client.chat.completions.create(
                    model="gpt-4.1-nano",
                    messages=[
                        {"role": "system", "content": "You are a professional translator. Translate the following Chinese text to English directly."},
                        {"role": "user", "content": str(text)}
                    ],
                    temperature=0.1
                )
                return text, response.choices[0].message.content.strip()
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "rate limit" in error_msg:
                    await asyncio.sleep(5 * (2 ** attempt))
                else:
                    if attempt == max_retries - 1:
                        return text, f"ERROR: {e}"
                    await asyncio.sleep(2 ** attempt)

async def main():
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("OPENAI_API_KEY")
    client = AsyncOpenAI(api_key=api_key)

    input_path = '/kaggle/input/datasets/shenanigannnnnn/week18-chinese-cleaned/week_18_cleaned_chinese_text_for_translation.pkl'
    output_path = '/kaggle/working/FINAL_week_18_translated_texts.csv'
    partial_path = '/kaggle/input/datasets/shenanigannnnnn/partial-translated-week18-final2/Partial_Translated_Week18_Final2.csv'

    with open(input_path, 'rb') as file:
        data = pickle.load(file)

    data = [text for text in data if str(text).strip() != "转发微博"]

    if os.path.exists(partial_path):
        existing_df = pd.read_csv(partial_path)
        start_index = len(existing_df)
        existing_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    else:
        start_index = 0

    target_limit = 1000000
    texts_to_translate = data[start_index:target_limit]
    semaphore = asyncio.Semaphore(10)
    chunk_size = 50
    print_count = 0
    delay_seconds = 60.0 / 450.0

    for i in range(0, len(texts_to_translate), chunk_size):
        chunk = texts_to_translate[i:i + chunk_size]
        tasks = []

        for text in chunk:
            task = asyncio.create_task(translate_text(text, client, semaphore))
            tasks.append(task)
            await asyncio.sleep(delay_seconds)

        results = []
        for coro in tqdm(asyncio.as_completed(tasks), total=len(chunk), desc=f"Translating chunk {i//chunk_size + 1}"):
            original, translated = await coro
            results.append({'translated': translated})

            if print_count < 50:
                print(f"Original: {original}\nTranslated: {translated}\n{'-'*50}")
                print_count += 1

        df = pd.DataFrame(results)
        df.to_csv(output_path, mode='a', index=False, header=not os.path.exists(output_path), encoding='utf-8-sig')

await main()

Translating chunk 1:   0%|          | 0/50 [00:00<?, ?it/s]

Translating chunk 1: 100%|██████████| 50/50 [00:00<00:00, 252.09it/s]

Translating chunk 1: 100%|██████████| 50/50 [00:00<00:00, 250.67it/s]

Original: 哈雷传人现身北京车展源自美国的世界顶级休闲摩托车品牌哈雷戴维森携早期定制哈雷摩托车亮相北京车展公司创始人之一的曾孙比尔戴维森先生bill davidson也亲临发布会现场并启动寻找最早的哈雷活动
Translated: The heir of Harley appeared at the Beijing Auto Show. The world-class leisure motorcycle brand Harley-Davidson from the United States showcased early customized Harley motorcycles at the Beijing Auto Show. One of the company's founders' great-grandson, Mr. Bill Davidson, also attended the launch event in person and initiated the search for the earliest Harley.
--------------------------------------------------
Original: 男生权利反转时当看到两个很丑的浓妆女生牵手走上来舞台我要是男嘉宾立马就得崩溃了哈哈
Translated: When the male's rights are reversed and I see two very ugly heavily made-up girls holding hands walking onto the stage, I would immediately break down if I were a male guest, haha.
--------------------------------------------------
Original: 变废为宝低碳生活我在三国来了中合成了春秋左氏传i can play
Translated: Turning waste into treasure, low-carbon living. I came to the Three Kingdoms and synthesized the Spring and Autumn

Translating chunk 2:   0%|          | 0/50 [00:00<?, ?it/s]

...

Translating chunk 4074:  92%|█████████▏| 46/50 [00:00<00:00, 180.96it/s]

Translating chunk 4074: 100%|██████████| 50/50 [00:00<00:00, 69.25it/s] 

Translating chunk 4075:   0%|          | 0/50 [00:00<?, ?it/s]

Translating chunk 4075:  94%|█████████▍| 47/50 [00:00<00:00, 176.69it/s]

Translating chunk 4075: 100%|██████████| 50/50 [00:00<00:00, 106.42it/s]